<a href="https://colab.research.google.com/github/Bl00df1s7/GLDRUBF-Sentry/blob/main/%D0%91%D0%BE%D0%B5%D0%B2%D0%BE%D0%B9_GLDRUBF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install t-tech-investments --index-url https://opensource.tbank.ru/api/v4/projects/238/packages/pypi/simple

Looking in indexes: https://opensource.tbank.ru/api/v4/projects/238/packages/pypi/simple


In [14]:
# ============================================================
# 01 — IMPORTS
# ============================================================

import pandas as pd
import numpy as np

from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

from google.colab import userdata

from t_tech.invest import Client
from t_tech.invest import CandleInterval

print("Imports OK")

Imports OK


In [15]:
# ============================================================
# 02 — T-INVEST API
# ============================================================

TOKEN = userdata.get("T-Sandapi")

if not TOKEN:
    raise RuntimeError(
        "Secret T-Sandapi не найден"
    )

print("Token получен")

client = Client(TOKEN)

print("T-Invest Client создан")

Token получен
T-Invest Client создан


In [16]:
# ============================================================
# 03 — GLDRUBF INSTRUMENT
# ============================================================

TARGET_TICKER = "GLDRUBF"

with Client(TOKEN) as services:

    response = services.instruments.futures()

    futures = response.instruments


instrument = None

for x in futures:

    if x.ticker.upper() == TARGET_TICKER:

        instrument = x
        break


if instrument is None:

    raise RuntimeError(
        f"Фьючерс {TARGET_TICKER} не найден"
    )


UID = instrument.uid
FIGI = instrument.figi
CLASS_CODE = instrument.class_code
LOT = instrument.lot
MIN_PRICE_INCREMENT = instrument.min_price_increment


print("=== INSTRUMENT ===")

print(
    f"Ticker:       {instrument.ticker}"
)

print(
    f"Name:         {instrument.name}"
)

print(
    f"UID:          {UID}"
)

print(
    f"FIGI:         {FIGI}"
)

print(
    f"Class code:   {CLASS_CODE}"
)

print(
    f"Lot:          {LOT}"
)

print(
    f"Min tick:     {MIN_PRICE_INCREMENT}"
)

/tmp/ipykernel_84548/3903992157.py:9: DeprecatedWarning: futures is deprecated as of 1.0.0.
  response = services.instruments.futures()


=== INSTRUMENT ===
Ticker:       GLDRUBF
Name:         GLDRUBF Золото (rub)
UID:          b347fe28-0d2a-45bf-b3bd-cda8a6ac64e6
FIGI:         FUTGLDRUBF00
Class code:   SPBFUT
Lot:          1
Min tick:     Quotation(units=0, nano=100000000)


In [17]:
# ============================================================
# 04 — STRATEGY CONFIG
# ============================================================

TIMEFRAME = "4H"

# ------------------------------------------------------------
# ENTRY
# ------------------------------------------------------------

DONCHIAN_LEN = 20


# ------------------------------------------------------------
# VOLATILITY
# ------------------------------------------------------------

ATR_LEN = 14


# ------------------------------------------------------------
# RISK MANAGEMENT
# ------------------------------------------------------------

SL_ATR = 3.0

TP_PCT = 0.07

BE_PCT = 0.02


# ------------------------------------------------------------
# PARABOLIC SAR
# ------------------------------------------------------------

SAR_START = 0.03
SAR_INC = 0.02
SAR_MAX = 0.20


print("=== STRATEGY CONFIG ===")

print(
    f"Instrument:    {TARGET_TICKER}"
)

print(
    f"Timeframe:     {TIMEFRAME}"
)

print()

print("ENTRY")

print(
    f"Donchian:      {DONCHIAN_LEN}"
)

print()

print("VOLATILITY")

print(
    f"ATR:           {ATR_LEN}"
)

print()

print("RISK")

print(
    f"SL:            {SL_ATR} ATR"
)

print(
    f"TP:            {TP_PCT * 100:.1f}%"
)

print(
    f"BE trigger:    {BE_PCT * 100:.1f}%"
)

print()

print("SAR")

print(
    f"Start:         {SAR_START}"
)

print(
    f"Increment:     {SAR_INC}"
)

print(
    f"Maximum:       {SAR_MAX}"
)

=== STRATEGY CONFIG ===
Instrument:    GLDRUBF
Timeframe:     4H

ENTRY
Donchian:      20

VOLATILITY
ATR:           14

RISK
SL:            3.0 ATR
TP:            7.0%
BE trigger:    2.0%

SAR
Start:         0.03
Increment:     0.02
Maximum:       0.2


In [18]:
# ============================================================
# 05 — CANDLE LOADER
# ============================================================

def quotation_to_float(value):
    """
    Безопасное преобразование Quotation в float.
    """

    if value is None:
        return np.nan


    if isinstance(
        value,
        (int, float, np.number)
    ):

        return float(value)


    if (
        hasattr(value, "units")
        and hasattr(value, "nano")
    ):

        return (
            float(value.units)
            + float(value.nano)
            / 1_000_000_000
        )


    if hasattr(value, "value"):

        return float(value.value)


    return float(value)


def candle_to_row(candle):

    return {

        "time": candle.time,

        "open": quotation_to_float(
            candle.open
        ),

        "high": quotation_to_float(
            candle.high
        ),

        "low": quotation_to_float(
            candle.low
        ),

        "close": quotation_to_float(
            candle.close
        ),

        "volume": candle.volume,
    }


def load_recent_candles(
    uid,
    candles_count=200
):

    now_utc = datetime.now(
        timezone.utc
    )


    # 4H = 6 свечей в сутки.
    # Добавляем запас истории.

    days = int(
        candles_count / 6
    ) + 10


    start_date = (
        now_utc
        - timedelta(days=days)
    )


    rows = []

    current = start_date

    chunk = timedelta(days=90)


    while current < now_utc:

        chunk_end = min(
            current + chunk,
            now_utc
        )


        with Client(TOKEN) as services:

            response = (
                services.market_data.get_candles(

                    instrument_id=uid,

                    from_=current,

                    to=chunk_end,

                    interval=(
                        CandleInterval
                        .CANDLE_INTERVAL_4_HOUR
                    ),
                )
            )


        rows.extend(

            candle_to_row(candle)

            for candle
            in response.candles
        )


        current = chunk_end


    df = pd.DataFrame(rows)


    if df.empty:

        return df


    df["time"] = pd.to_datetime(
        df["time"],
        utc=True
    )


    df = (

        df

        .drop_duplicates("time")

        .sort_values("time")

        .reset_index(drop=True)
    )


    return (
        df
        .tail(candles_count)
        .reset_index(drop=True)
    )

In [19]:
# ============================================================
# 06 — INDICATORS
# ============================================================

def calculate_atr(
    df,
    length
):

    high = df["high"]
    low = df["low"]
    close = df["close"]

    prev_close = close.shift(1)


    tr = pd.concat(

        [

            high - low,

            (
                high - prev_close
            ).abs(),

            (
                low - prev_close
            ).abs(),

        ],

        axis=1

    ).max(axis=1)


    return (
        tr
        .rolling(length)
        .mean()
    )


def prepare_indicators(df):

    data = (
        df
        .copy()
        .reset_index(drop=True)
    )


    # ========================================================
    # ATR
    # ========================================================

    data["atr"] = calculate_atr(
        data,
        ATR_LEN
    )


    # ========================================================
    # DONCHIAN
    #
    # Только предыдущие свечи.
    # ========================================================

    data["donchian_upper"] = (

        data["high"]

        .rolling(DONCHIAN_LEN)

        .max()

        .shift(1)
    )


    data["donchian_lower"] = (

        data["low"]

        .rolling(DONCHIAN_LEN)

        .min()

        .shift(1)
    )


    # ========================================================
    # ENTRY SIGNALS
    # ========================================================

    data["long_signal"] = (

        data["close"]
        > data["donchian_upper"]
    )


    data["short_signal"] = (

        data["close"]
        < data["donchian_lower"]
    )


    return data

In [20]:
# ============================================================
# 07 — PARABOLIC SAR
# ============================================================

def calculate_sar(
    df,
    start,
    inc,
    maximum
):

    high = df["high"].to_numpy()
    low = df["low"].to_numpy()
    close = df["close"].to_numpy()

    n = len(df)


    sar = np.full(
        n,
        np.nan
    )

    ep = np.full(
        n,
        np.nan
    )

    af = np.full(
        n,
        np.nan
    )

    trend = np.ones(
        n,
        dtype=int
    )


    if n == 0:

        return sar, trend


    sar[0] = close[0]
    ep[0] = close[0]
    af[0] = start
    trend[0] = 1


    for i in range(
        1,
        n
    ):

        prev_sar = sar[i - 1]
        prev_ep = ep[i - 1]
        prev_af = af[i - 1]
        prev_trend = trend[i - 1]


        # ====================================================
        # UP TREND
        # ====================================================

        if prev_trend == 1:

            current_sar = (

                prev_sar

                + prev_af
                * (
                    prev_ep
                    - prev_sar
                )
            )


            if i >= 2:

                current_sar = min(

                    current_sar,

                    low[i - 1],

                    low[i - 2]
                )

            else:

                current_sar = min(

                    current_sar,

                    low[i - 1]
                )


            # ------------------------------------------------
            # REVERSAL
            # ------------------------------------------------

            if low[i] < current_sar:

                trend[i] = -1

                sar[i] = prev_ep

                ep[i] = low[i]

                af[i] = start


            # ------------------------------------------------
            # CONTINUE UP
            # ------------------------------------------------

            else:

                trend[i] = 1

                sar[i] = current_sar


                if high[i] > prev_ep:

                    ep[i] = high[i]

                    af[i] = min(

                        prev_af + inc,

                        maximum
                    )

                else:

                    ep[i] = prev_ep

                    af[i] = prev_af


        # ====================================================
        # DOWN TREND
        # ====================================================

        else:

            current_sar = (

                prev_sar

                + prev_af
                * (
                    prev_ep
                    - prev_sar
                )
            )


            if i >= 2:

                current_sar = max(

                    current_sar,

                    high[i - 1],

                    high[i - 2]
                )

            else:

                current_sar = max(

                    current_sar,

                    high[i - 1]
                )


            # ------------------------------------------------
            # REVERSAL
            # ------------------------------------------------

            if high[i] > current_sar:

                trend[i] = 1

                sar[i] = prev_ep

                ep[i] = high[i]

                af[i] = start


            # ------------------------------------------------
            # CONTINUE DOWN
            # ------------------------------------------------

            else:

                trend[i] = -1

                sar[i] = current_sar


                if low[i] < prev_ep:

                    ep[i] = low[i]

                    af[i] = min(

                        prev_af + inc,

                        maximum
                    )

                else:

                    ep[i] = prev_ep

                    af[i] = prev_af


    return sar, trend

In [21]:
# ============================================================
# 08 — MARKET DATA
# ============================================================

df_raw = load_recent_candles(
    UID,
    candles_count=200
)


if df_raw.empty:

    raise RuntimeError(
        "Не удалось получить свечи GLDRUBF"
    )


df = prepare_indicators(
    df_raw
)


df["sar"], df["sar_trend"] = calculate_sar(

    df,

    SAR_START,
    SAR_INC,
    SAR_MAX
)


# ============================================================
# CURRENT TIME
# ============================================================

now_utc = datetime.now(
    timezone.utc
)

MSK = ZoneInfo(
    "Europe/Moscow"
)


# ============================================================
# FIND LAST CLOSED 4H CANDLE
#
# T-Invest 4H свечи выровнены по UTC.
#
# Если candle.time = начало свечи,
# она считается закрытой только после:
#
# candle.time + 4 часа <= now
#
# ============================================================

CANDLE_DURATION = timedelta(
    hours=4
)


df["candle_close_time"] = (
    df["time"]
    + CANDLE_DURATION
)


closed_candidates = df[
    df["candle_close_time"]
    <= now_utc
].copy()


if closed_candidates.empty:

    raise RuntimeError(
        "Не найдена ни одна закрытая 4H свеча"
    )


closed = (
    closed_candidates
    .iloc[-1]
)


closed_index = (
    closed_candidates.index[-1]
)


if closed_index == 0:

    raise RuntimeError(
        "Недостаточно истории для предыдущей закрытой свечи"
    )


previous_closed = (
    df.loc[
        closed_index - 1
    ]
)


# ============================================================
# CURRENT PRICE
# ============================================================

with Client(TOKEN) as services:

    response = (
        services.market_data.get_last_prices(

            instrument_id=[UID]
        )
    )


if not response.last_prices:

    raise RuntimeError(
        "Не удалось получить текущую цену GLDRUBF"
    )


current_price = quotation_to_float(

    response
    .last_prices[0]
    .price
)


# ============================================================
# DISPLAY
# ============================================================

print("=== MARKET DATA ===")

print(
    f"Current UTC:       {now_utc}"
)

print(
    f"Current MSK:       "
    f"{now_utc.astimezone(MSK)}"
)

print()

print(
    f"Last closed 4H:    "
    f"{closed['time'].astimezone(MSK)}"
)

print(
    f"Candle close UTC:  "
    f"{closed['candle_close_time']}"
)

print(
    f"Candle close MSK:  "
    f"{closed['candle_close_time'].astimezone(MSK)}"
)

print()

print(
    f"Open:              {closed['open']:.2f}"
)

print(
    f"High:              {closed['high']:.2f}"
)

print(
    f"Low:               {closed['low']:.2f}"
)

print(
    f"Close:             {closed['close']:.2f}"
)

print(
    f"Current price:     {current_price:.2f}"
)

print(
    f"ATR:               {closed['atr']:.2f}"
)

print(
    f"SAR:               {closed['sar']:.2f}"
)

print(
    f"SAR trend:         "
    f"{'LONG' if closed['sar_trend'] == 1 else 'SHORT'}"
)

/tmp/ipykernel_84548/3016162735.py:110: DeprecatedWarning: get_candles is deprecated as of 1.0.0.
  services.market_data.get_candles(
/tmp/ipykernel_84548/3611277291.py:114: DeprecatedWarning: get_last_prices is deprecated as of 1.0.0.
  services.market_data.get_last_prices(


=== MARKET DATA ===
Current UTC:       2026-08-12 13:01:12.376183+00:00
Current MSK:       2026-08-12 16:01:12.376183+03:00

Last closed 4H:    2026-08-12 11:00:00+03:00
Candle close UTC:  2026-08-12 12:00:00+00:00
Candle close MSK:  2026-08-12 15:00:00+03:00

Open:              11647.80
High:              11715.20
Low:               11647.80
Close:             11688.10
Current price:     11711.00
ATR:               95.71
SAR:               11402.84
SAR trend:         LONG


In [22]:
# ============================================================
# 09 — ALL ACCOUNTS
# ============================================================

with Client(TOKEN) as services:

    accounts_response = (
        services.users.get_accounts()
    )


accounts = (
    accounts_response.accounts
)


if not accounts:

    raise RuntimeError(
        "Для этого токена не найдено ни одного счёта"
    )


print("=== ACCOUNTS ===")

print(
    f"Всего счетов: {len(accounts)}"
)

print()


for i, account in enumerate(
    accounts,
    start=1
):

    print(
        f"{i}. "
        f"ID: {account.id} | "
        f"Name: {account.name} | "
        f"Type: {account.type} | "
        f"Status: {account.status}"
    )

/tmp/ipykernel_84548/711042713.py:8: DeprecatedWarning: get_accounts is deprecated as of 1.0.0.
  services.users.get_accounts()


=== ACCOUNTS ===
Всего счетов: 10

1. ID: 2009052016 | Name: Брокерский счёт | Type: 1 | Status: 2
2. ID: 2163652650 | Name: Тест секьюр  | Type: 1 | Status: 2
3. ID: 2040428803 | Name: ИИС | Type: 2 | Status: 2
4. ID: 2113327027 | Name: Road to 1kk | Type: 1 | Status: 2
5. ID: 2035914981 | Name: Инвесткопилка | Type: 3 | Status: 2
6. ID: 2093956148 | Name: Счет | Type: 1 | Status: 2
7. ID: 2146139894 | Name: Портфельное инвестирование 1 | Type: 1 | Status: 2
8. ID: 2218152205 | Name: Смарт-счет | Type: 7 | Status: 2
9. ID: 2226775427 | Name: Счет под ключ | Type: 4 | Status: 2
10. ID: 2023149547 | Name: Счет под ключ 1 | Type: 4 | Status: 2


In [23]:
# ============================================================
# 10 — PORTFOLIO POSITION
# ============================================================

print()
print("=== GLDRUBF POSITIONS ===")
print()

# ============================================================
# DEFAULT STATE
# ============================================================

position_qty = 0.0
position_direction = "NONE"

POSITION_ACCOUNT_ID = None
POSITION_ACCOUNT_NAME = None

gldrubf_position = None


# ============================================================
# LOAD ALL ACCOUNTS
# ============================================================

with Client(TOKEN) as services:

    accounts_response = (
        services.users.get_accounts()
    )

    accounts = accounts_response.accounts

    if not accounts:

        raise RuntimeError(
            "Для этого токена не найдено ни одного счёта"
        )

    print(
        f"Найдено счетов: {len(accounts)}"
    )

    # ========================================================
    # CHECK EVERY ACCOUNT
    # ========================================================

    for account in accounts:

        account_id = account.id

        print()
        print("=" * 60)

        print(
            f"ACCOUNT: {account_id}"
        )

        print(
            f"NAME:    {account.name}"
        )

        print(
            f"TYPE:    {account.type}"
        )

        print(
            f"STATUS:  {account.status}"
        )

        try:

            positions_response = (
                services.operations.get_positions(
                    account_id=account_id
                )
            )

        except Exception as e:

            print(
                f"⚠️ Не удалось получить позиции: {e}"
            )

            print(
                "Счёт пропускаем."
            )

            continue


        futures_positions = (
            positions_response.futures
        )

        print()
        print("Futures positions:")
        print(futures_positions)


        # ====================================================
        # SEARCH GLDRUBF
        # ====================================================

        for position in futures_positions:

            if (
                position.ticker.upper()
                != TARGET_TICKER
            ):
                continue


            balance = float(
                position.balance
            )


            print()
            print("🎯 GLDRUBF FOUND")

            print(
                f"Ticker:          {position.ticker}"
            )

            print(
                f"Instrument UID:  {position.instrument_uid}"
            )

            print(
                f"Balance:         {balance}"
            )

            print(
                f"Blocked:         {position.blocked}"
            )


            # =================================================
            # POSITION EXISTS
            # =================================================

            if balance != 0:

                gldrubf_position = position

                position_qty = balance

                POSITION_ACCOUNT_ID = account_id

                POSITION_ACCOUNT_NAME = (
                    account.name
                )

                if balance > 0:

                    position_direction = "LONG"

                else:

                    position_direction = "SHORT"


                print()
                print(
                    "🎯 GLDRUBF POSITION FOUND"
                )

                print(
                    f"Account:         "
                    f"{POSITION_ACCOUNT_ID}"
                )

                print(
                    f"Account name:    "
                    f"{POSITION_ACCOUNT_NAME}"
                )

                print(
                    f"Ticker:          "
                    f"{position.ticker}"
                )

                print(
                    f"Quantity:        "
                    f"{position_qty}"
                )

                print(
                    f"Direction:       "
                    f"{position_direction}"
                )

                print(
                    f"Blocked:         "
                    f"{position.blocked}"
                )

                # ---------------------------------------------
                # ВАЖНО:
                # нашли реальную позицию.
                # Больше её не перезаписываем.
                # ---------------------------------------------

                break


        # ====================================================
        # POSITION FOUND → STOP CHECKING ACCOUNTS
        # ====================================================

        if gldrubf_position is not None:

            break


# ============================================================
# FINAL POSITION STATE
# ============================================================

print()
print("=" * 60)
print("FINAL POSITION STATE")
print("=" * 60)

print(
    f"Account:    {POSITION_ACCOUNT_ID}"
)

print(
    f"Account:    {POSITION_ACCOUNT_NAME}"
)

print(
    f"Quantity:   {position_qty}"
)

print(
    f"Direction:  {position_direction}"
)

print()


if position_direction == "NONE":

    print(
        "⚪ FINAL: NO GLDRUBF POSITION"
    )

else:

    print(
        f"🟢 FINAL: GLDRUBF {position_direction}"
    )


=== GLDRUBF POSITIONS ===



/tmp/ipykernel_84548/2527008147.py:29: DeprecatedWarning: get_accounts is deprecated as of 1.0.0.
  services.users.get_accounts()


Найдено счетов: 10

ACCOUNT: 2009052016
NAME:    Брокерский счёт
TYPE:    1
STATUS:  2

Futures positions:
[]

ACCOUNT: 2163652650
NAME:    Тест секьюр 
TYPE:    1
STATUS:  2


/tmp/ipykernel_84548/2527008147.py:74: DeprecatedWarning: get_positions is deprecated as of 1.0.0.
  services.operations.get_positions(



Futures positions:
[PositionsFutures(figi='FUTGLDRUBF00', blocked=0, balance=3, position_uid='1f9dd5a6-5b24-485e-bc8b-78ee2ca3ea23', instrument_uid='b347fe28-0d2a-45bf-b3bd-cda8a6ac64e6', ticker='GLDRUBF'), PositionsFutures(figi='FUTCNYRUBF00', blocked=0, balance=-1, position_uid='20cf3eec-7277-4e9f-a392-c1641bf32821', instrument_uid='c300543d-aa18-4249-b110-615409dde036', ticker='CNYRUBF')]

🎯 GLDRUBF FOUND
Ticker:          GLDRUBF
Instrument UID:  b347fe28-0d2a-45bf-b3bd-cda8a6ac64e6
Balance:         3.0
Blocked:         0

🎯 GLDRUBF POSITION FOUND
Account:         2163652650
Account name:    Тест секьюр 
Ticker:          GLDRUBF
Quantity:        3.0
Direction:       LONG
Blocked:         0

FINAL POSITION STATE
Account:    2163652650
Account:    Тест секьюр 
Quantity:   3.0
Direction:  LONG

🟢 FINAL: GLDRUBF LONG


In [24]:
# ============================================================
# 11 — POSITION STATE
# ============================================================

# ============================================================
# POSITION ALREADY DETERMINED IN CELL 10
# ============================================================

if gldrubf_position is None:

    # Позиции GLDRUBF нет
    position_direction = "NONE"
    position_qty = 0.0

    entry_price = np.nan
    entry_atr = np.nan

    position_account_id = None
    position_account_name = None

else:

    # Реальная позиция найдена
    position_qty = float(
        gldrubf_position.balance
    )

    position_direction = (
        "LONG"
        if position_qty > 0
        else "SHORT"
    )

    position_account_id = (
        POSITION_ACCOUNT_ID
    )

    position_account_name = (
        POSITION_ACCOUNT_NAME
    )

    # --------------------------------------------------------
    # ВАЖНО:
    #
    # T-Invest PositionsFutures содержит количество,
    # но НЕ содержит цену входа.
    #
    # Поэтому пока entry_price неизвестна.
    # --------------------------------------------------------

    entry_price = np.nan

    # ATR берём с последней закрытой 4H свечи
    entry_atr = float(
        closed["atr"]
    )


print("=== POSITION STATE ===")

if position_direction == "NONE":

    print(
        "Position: NONE"
    )

else:

    print(
        f"Position:       {position_direction}"
    )

    print(
        f"Account:        {position_account_name}"
    )

    print(
        f"Account ID:     {position_account_id}"
    )

    print(
        f"Quantity:       {position_qty}"
    )

    print(
        f"Entry price:    {entry_price}"
    )

    print(
        f"Entry ATR:      {entry_atr:.2f}"
    )

=== POSITION STATE ===
Position:       LONG
Account:        Тест секьюр 
Account ID:     2163652650
Quantity:       3.0
Entry price:    nan
Entry ATR:      95.71


In [25]:
# ============================================================
# 11 — POSITION STATE
# ============================================================

# Состояние уже определено в ячейке 10.
# НИЧЕГО повторно не ищем через all_gldrubf_positions.

if gldrubf_position is None:

    position_direction = "NONE"
    position_qty = 0.0

    position_account_id = None
    position_account_name = None

    entry_price = np.nan
    entry_atr = np.nan

else:

    position_qty = float(
        gldrubf_position.balance
    )

    position_direction = (
        "LONG"
        if position_qty > 0
        else "SHORT"
    )

    position_account_id = (
        POSITION_ACCOUNT_ID
    )

    position_account_name = (
        POSITION_ACCOUNT_NAME
    )

    # --------------------------------------------------------
    # Получаем среднюю цену позиции через portfolio
    # --------------------------------------------------------

    entry_price = np.nan

    with Client(TOKEN) as services:

        portfolio = (
            services.operations.get_portfolio(
                account_id=position_account_id
            )
        )

    for portfolio_position in portfolio.positions:

        if (
            portfolio_position.figi
            == gldrubf_position.figi
        ):

            entry_price = quotation_to_float(
                portfolio_position.average_position_price
            )

            break

    if np.isnan(entry_price):

        print()
        print(
            "⚠️ Не удалось получить "
            "среднюю цену GLDRUBF"
        )

    # ATR для уровней
    entry_atr = float(
        closed["atr"]
    )


# ============================================================
# DISPLAY
# ============================================================

print()
print("=== POSITION STATE ===")

print(
    f"Account:    {position_account_id}"
)

print(
    f"Account:    {position_account_name}"
)

print(
    f"Quantity:   {position_qty}"
)

print(
    f"Direction:  {position_direction}"
)

print(
    f"Entry:      "
    f"{entry_price:.2f}"
    if not np.isnan(entry_price)
    else "Entry:      N/A"
)

/tmp/ipykernel_84548/190895587.py:48: DeprecatedWarning: get_portfolio is deprecated as of 1.0.0.
  services.operations.get_portfolio(



=== POSITION STATE ===
Account:    2163652650
Account:    Тест секьюр 
Quantity:   3.0
Direction:  LONG
Entry:      11659.70


In [26]:
# ============================================================
# 12 — POSITION LEVELS
# ============================================================

sl_price = np.nan
tp_price = np.nan
be_trigger = np.nan
sar_price = float(closed["sar"])


if position_direction in ("LONG", "SHORT"):

    if np.isnan(entry_price):

        print(
            "⚠️ Нет средней цены позиции — "
            "уровни не рассчитываем"
        )

    else:

        entry_price = float(entry_price)
        entry_atr = float(entry_atr)

        # ====================================================
        # LONG
        # ====================================================

        if position_direction == "LONG":

            sl_price = (
                entry_price
                - entry_atr * SL_ATR
            )

            tp_price = (
                entry_price
                * (1 + TP_PCT)
            )

            be_trigger = (
                entry_price
                * (1 + BE_PCT)
            )

        # ====================================================
        # SHORT
        # ====================================================

        else:

            sl_price = (
                entry_price
                + entry_atr * SL_ATR
            )

            tp_price = (
                entry_price
                * (1 - TP_PCT)
            )

            be_trigger = (
                entry_price
                * (1 - BE_PCT)
            )


print()
print("=== LEVELS ===")

if position_direction == "NONE":

    print("Position: NONE")

elif position_direction == "MULTIPLE":

    print("Position: MULTIPLE ACCOUNTS")

elif np.isnan(entry_price):

    print(
        f"Position: {position_direction}"
    )

    print(
        "Entry:    N/A"
    )

else:

    print(
        f"Account:       {position_account_name}"
    )

    print(
        f"Direction:     {position_direction}"
    )

    print(
        f"Quantity:      {position_qty}"
    )

    print(
        f"Entry:         {entry_price:.2f}"
    )

    print(
        f"SL:            {sl_price:.2f}"
    )

    print(
        f"TP:            {tp_price:.2f}"
    )

    print(
        f"BE trigger:    {be_trigger:.2f}"
    )

    print(
        f"SAR:           {sar_price:.2f}"
    )


=== LEVELS ===
Account:       Тест секьюр 
Direction:     LONG
Quantity:      3.0
Entry:         11659.70
SL:            11372.58
TP:            12475.88
BE trigger:    11892.89
SAR:           11402.84


In [27]:
# ============================================================
# 13 — SENTRY DECISION
# ============================================================

print()
print("=" * 70)
print("GLDRUBF SENTRY")
print("=" * 70)

print()

print(
    f"Time:           "
    f"{now_utc.astimezone(MSK)}"
)

print(
    f"Current price:  "
    f"{current_price:.2f}"
)

print(
    f"Closed 4H:      "
    f"{closed['time'].astimezone(MSK)}"
)

print(
    f"Closed price:   "
    f"{closed['close']:.2f}"
)

print()


# ============================================================
# POSITION
# ============================================================

print("POSITION")


if position_direction == "NONE":

    print(
        "⚪ NO POSITION"
    )


elif position_direction == "MULTIPLE":

    print(
        "⚠️ MULTIPLE POSITIONS"
    )

    print(
        "Sentry action: NO ACTION"
    )


else:

    if position_direction == "LONG":

        print(
            "🟢 LONG"
        )

    else:

        print(
            "🔴 SHORT"
        )


    print(
        f"Account:        "
        f"{position_account_name}"
    )

    print(
        f"Quantity:       "
        f"{position_qty}"
    )

    print(
        f"Entry:          "
        f"{entry_price:.2f}"
    )

    print(
        f"Current:        "
        f"{current_price:.2f}"
    )


print()


# ============================================================
# NO POSITION → ENTRY SIGNAL
# ============================================================

if position_direction == "NONE":

    print("SIGNAL")


    if closed["long_signal"]:

        entry_reference = (
            current_price
        )


        signal_sl = (

            entry_reference

            - closed["atr"]
            * SL_ATR
        )


        signal_tp = (

            entry_reference

            * (1 + TP_PCT)
        )


        signal_be = (

            entry_reference

            * (1 + BE_PCT)
        )


        print(
            "🟢 LONG ENTRY"
        )

        print(
            f"Entry reference: "
            f"{entry_reference:.2f}"
        )

        print(
            f"SL:              "
            f"{signal_sl:.2f}"
        )

        print(
            f"TP:              "
            f"{signal_tp:.2f}"
        )

        print(
            f"BE trigger:      "
            f"{signal_be:.2f}"
        )


    elif closed["short_signal"]:

        entry_reference = (
            current_price
        )


        signal_sl = (

            entry_reference

            + closed["atr"]
            * SL_ATR
        )


        signal_tp = (

            entry_reference

            * (1 - TP_PCT)
        )


        signal_be = (

            entry_reference

            * (1 - BE_PCT)
        )


        print(
            "🔴 SHORT ENTRY"
        )

        print(
            f"Entry reference: "
            f"{entry_reference:.2f}"
        )

        print(
            f"SL:              "
            f"{signal_sl:.2f}"
        )

        print(
            f"TP:              "
            f"{signal_tp:.2f}"
        )

        print(
            f"BE trigger:      "
            f"{signal_be:.2f}"
        )


    else:

        print(
            "⚪ NO ENTRY SIGNAL"
        )


# ============================================================
# MULTIPLE POSITIONS
# ============================================================

elif position_direction == "MULTIPLE":

    print(
        "ACTION"
    )

    print(
        "⚠️ NO ACTION — "
        "multiple GLDRUBF positions found"
    )


# ========================================================
# EXISTING POSITION → EXIT CHECK
# ========================================================

else:

    print("LEVELS")

    if np.isnan(entry_price):

        print(
            "⚠️ Средняя цена позиции неизвестна."
        )

        print(
            "⚠️ EXIT CHECK НЕ ВЫПОЛНЯЕМ."
        )

        print(
            "ACTION"
        )

        print(
            "⚠️ NO ACTION — "
            "не удалось определить Entry"
        )

    else:

        print(
            f"SL:             "
            f"{sl_price:.2f}"
        )

        print(
            f"TP:             "
            f"{tp_price:.2f}"
        )

        print(
            f"BE trigger:     "
            f"{be_trigger:.2f}"
        )

        print(
            f"SAR:            "
            f"{sar_price:.2f}"
        )

        print()

        # ==================================================
        # EXIT CONDITIONS
        # ==================================================

        if position_direction == "LONG":

            hit_sl = (
                current_price <= sl_price
            )

            hit_tp = (
                current_price >= tp_price
            )

            hit_be = (
                current_price >= be_trigger
            )

            sar_exit = (
                closed["sar_trend"] == -1
                and
                previous_closed["sar_trend"] != -1
            )

        else:

            hit_sl = (
                current_price >= sl_price
            )

            hit_tp = (
                current_price <= tp_price
            )

            hit_be = (
                current_price <= be_trigger
            )

            sar_exit = (
                closed["sar_trend"] == 1
                and
                previous_closed["sar_trend"] != 1
            )

        # ==================================================
        # ACTION
        # ==================================================

        print("ACTION")

        if hit_sl:

            print(
                "🔴 EXIT — SL"
            )

        elif hit_tp:

            print(
                "🟢 EXIT — TP"
            )

        elif sar_exit:

            print(
                "🟡 EXIT — SAR"
            )

        elif hit_be:

            print(
                "🟡 BE TRIGGER REACHED"
            )

        else:

            print(
                "🟢 HOLD"
            )


GLDRUBF SENTRY

Time:           2026-08-12 16:01:12.376183+03:00
Current price:  11711.00
Closed 4H:      2026-08-12 11:00:00+03:00
Closed price:   11688.10

POSITION
🟢 LONG
Account:        Тест секьюр 
Quantity:       3.0
Entry:          11659.70
Current:        11711.00

LEVELS
SL:             11372.58
TP:             12475.88
BE trigger:     11892.89
SAR:            11402.84

ACTION
🟢 HOLD


In [28]:
# ============================================================
# 14 — FINAL STATUS
# ============================================================

print()
print("=" * 70)
print("GLDRUBF SENTRY — FINAL STATUS")
print("=" * 70)

print()

print(
    f"Updated:        "
    f"{now_utc.astimezone(MSK)}"
)

print(
    f"Price:          "
    f"{current_price:.2f}"
)

print(
    f"Last closed 4H: "
    f"{closed['time'].astimezone(MSK)}"
)

print(
    f"Closed price:   "
    f"{closed['close']:.2f}"
)

print()


# ============================================================
# POSITION
# ============================================================

if position_direction == "NONE":

    print(
        "POSITION:       ⚪ NONE"
    )


elif position_direction == "MULTIPLE":

    print(
        "POSITION:       ⚠️ MULTIPLE"
    )

    for position in all_gldrubf_positions:

        print(
            f"                "
            f"{position['account_name']} | "
            f"{position['direction']} | "
            f"{position['quantity']}"
        )


else:

    position_icon = (
        "🟢"
        if position_direction == "LONG"
        else "🔴"
    )


    print(
        f"POSITION:       "
        f"{position_icon} "
        f"{position_direction}"
    )

    print(
        f"ACCOUNT:        "
        f"{position_account_name}"
    )

    print(
        f"QUANTITY:       "
        f"{position_qty}"
    )

    print(
        f"ENTRY:          "
        f"{entry_price:.2f}"
    )

    print(
        f"SL:             "
        f"{sl_price:.2f}"
    )

    print(
        f"TP:             "
        f"{tp_price:.2f}"
    )

    print(
        f"BE:             "
        f"{be_trigger:.2f}"
    )


print()


# ============================================================
# SIGNAL
# ============================================================

if (
    closed["long_signal"]
):

    print(
        "ENTRY SIGNAL:   🟢 LONG"
    )


elif (
    closed["short_signal"]
):

    print(
        "ENTRY SIGNAL:   🔴 SHORT"
    )


else:

    print(
        "ENTRY SIGNAL:   ⚪ NONE"
    )


print()


# ============================================================
# SAR
# ============================================================

print(
    f"SAR:            "
    f"{closed['sar']:.2f}"
)

print(
    f"SAR trend:      "
    f"{'LONG' if closed['sar_trend'] == 1 else 'SHORT'}"
)


print()


# ============================================================
# ACTION
# ============================================================

if position_direction == "MULTIPLE":

    print(
        "ACTION:         ⚠️ NO ACTION"
    )


elif position_direction == "NONE":

    if closed["long_signal"]:

        print(
            "ACTION:         🟢 OPEN LONG"
        )


    elif closed["short_signal"]:

        print(
            "ACTION:         🔴 OPEN SHORT"
        )


    else:

        print(
            "ACTION:         ⚪ WAIT"
        )


else:

    if hit_sl:

        print(
            "ACTION:         🔴 EXIT — SL"
        )


    elif hit_tp:

        print(
            "ACTION:         🟢 EXIT — TP"
        )


    elif sar_exit:

        print(
            "ACTION:         🟡 EXIT — SAR"
        )


    elif hit_be:

        print(
            "ACTION:         🟡 BE"
        )


    else:

        print(
            "ACTION:         🟢 HOLD"
        )


print()

print("=" * 70)


GLDRUBF SENTRY — FINAL STATUS

Updated:        2026-08-12 16:01:12.376183+03:00
Price:          11711.00
Last closed 4H: 2026-08-12 11:00:00+03:00
Closed price:   11688.10

POSITION:       🟢 LONG
ACCOUNT:        Тест секьюр 
QUANTITY:       3.0
ENTRY:          11659.70
SL:             11372.58
TP:             12475.88
BE:             11892.89

ENTRY SIGNAL:   🟢 LONG

SAR:            11402.84
SAR trend:      LONG

ACTION:         🟢 HOLD



# Телеграмм бот

In [29]:
# ============================================================
# TELEGRAM — SETUP
# ============================================================

import requests

BOT_TOKEN = userdata.get("BOT_TOKEN")

if not BOT_TOKEN:
    raise RuntimeError(
        "Secret BOT_TOKEN не найден"
    )


def telegram_get_chat_id():

    url = (
        f"https://api.telegram.org/bot"
        f"{BOT_TOKEN}/getUpdates"
    )

    response = requests.get(
        url,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    if not data.get("ok"):
        raise RuntimeError(
            f"Telegram API error: {data}"
        )

    updates = data.get("result", [])

    # Берём последнее сообщение с chat.id
    for update in reversed(updates):

        message = update.get("message")

        if message and message.get("chat"):

            return message["chat"]["id"]

    # Если сообщений нет — пробуем getMe для проверки токена
    me_url = (
        f"https://api.telegram.org/bot"
        f"{BOT_TOKEN}/getMe"
    )
    
    me_response = requests.get(me_url, timeout=10)
    me_response.raise_for_status()
    me_data = me_response.json()
    
    if me_data.get("ok"):
        bot_username = me_data["result"].get("username", "")
        print(f"⚠️ Бот @{bot_username} активен, но нет новых сообщений.")
        print(f"   1. Отправьте боту /start в Telegram")
        print(f"   2. Либо узнайте свой chat_id через @userinfobot")
        print(f"   3. Задайте TELEGRAM_CHAT_ID вручную ниже")
    else:
        print("⚠️ Не удалось проверить бота. Проверьте BOT_TOKEN.")

    raise RuntimeError(
        "❌ Telegram не вернул сообщений. "
        "Отправь боту /start и запусти ячейку снова, "
        "либо задай TELEGRAM_CHAT_ID вручную."
    )


# Попытка авто-получения, иначе — ручной режим
try:
    TELEGRAM_CHAT_ID = telegram_get_chat_id()
    print(
        f"✅ Telegram chat_id получен: "
        f"{TELEGRAM_CHAT_ID}"
    )
except RuntimeError as e:
    print(str(e))
    # Резервный вариант: укажите chat_id вручную здесь
    TELEGRAM_CHAT_ID = None  # Замените на ваш chat_id, например: 591958455
    
    if TELEGRAM_CHAT_ID is None:
        raise RuntimeError(
            "Укажите TELEGRAM_CHAT_ID вручную в ячейке выше (замените None на ваш числовой ID)"
        )
    else:
        print(f"✅ Используется ручной chat_id: {TELEGRAM_CHAT_ID}")

Telegram chat_id получен: 591958455


In [30]:
# ============================================================
# TELEGRAM — SEND MESSAGE
# ============================================================

def telegram_send(message):

    url = (
        f"https://api.telegram.org/bot"
        f"{BOT_TOKEN}/sendMessage"
    )

    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "parse_mode": "HTML",
    }

    response = requests.post(
        url,
        json=payload,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    if not data.get("ok"):

        raise RuntimeError(
            f"Telegram API error: {data}"
        )

    return data


print("Telegram функция готова")

Telegram функция готова


In [31]:
# ============================================================
# TELEGRAM — FINAL SENTRY STATUS
# ============================================================

import requests

BOT_TOKEN = userdata.get("BOT_TOKEN")

if not BOT_TOKEN:
    raise RuntimeError(
        "Secret BOT_TOKEN не найден"
    )


# ============================================================
# HELPERS
# ============================================================

def fmt_price(value):
    return f"{float(value):,.2f}".replace(",", " ")


# ============================================================
# MARKET
# ============================================================

updated_msk = now_utc.astimezone(MSK)
closed_msk = closed["time"].astimezone(MSK)


# ============================================================
# POSITION
# ============================================================

if position_direction == "NONE":

    position_block = (
        "⚪ <b>Нет позиции</b>"
    )

elif position_direction == "MULTIPLE":

    position_block = (
        "⚠️ <b>Несколько позиций</b>"
    )

else:

    position_icon = (
        "🟢"
        if position_direction == "LONG"
        else "🔴"
    )

    position_block = (
        f"{position_icon} "
        f"<b>{position_direction} × {position_qty:g}</b>\n"
        f"Вход:        {fmt_price(entry_price)}\n"
        f"SL:          {fmt_price(sl_price)}\n"
        f"TP:          {fmt_price(tp_price)}\n"
        f"BE:          {fmt_price(be_trigger)}"
    )


# ============================================================
# SIGNAL
# ============================================================

if closed["long_signal"]:

    signal_block = "🟢 <b>LONG</b>"

elif closed["short_signal"]:

    signal_block = "🔴 <b>SHORT</b>"

else:

    signal_block = "⚪ Нет сигнала"


# ============================================================
# SAR
# ============================================================

sar_trend = (
    "LONG"
    if closed["sar_trend"] == 1
    else "SHORT"
)

sar_icon = (
    "🟢"
    if closed["sar_trend"] == 1
    else "🔴"
)

sar_block = (
    f"{fmt_price(closed['sar'])} "
    f"· {sar_icon} {sar_trend}"
)


# ============================================================
# ACTION
# ============================================================

if position_direction == "MULTIPLE":

    action = "⚠️ NO ACTION"

elif position_direction == "NONE":

    if closed["long_signal"]:

        action = "🟢 OPEN LONG"

    elif closed["short_signal"]:

        action = "🔴 OPEN SHORT"

    else:

        action = "⚪ WAIT"

else:

    if hit_sl:

        action = "🔴 EXIT — SL"

    elif hit_tp:

        action = "🟢 EXIT — TP"

    elif sar_exit:

        action = "🟡 EXIT — SAR"

    elif hit_be:

        action = "🟡 BE"

    else:

        action = "🟢 HOLD"


# ============================================================
# MESSAGE
# ============================================================

telegram_message = (
    "🟢 <b>GLDRUBF SENTRY</b>\n"
    "\n"

    "💰 <b>Рынок</b>\n"
    f"Цена:        <b>{fmt_price(current_price)}</b>\n"
    f"Закрытие 4H: {fmt_price(closed['close'])}\n"
    f"Свеча:       {closed_msk.strftime('%d.%m.%Y %H:%M')} MSK\n"
    "\n"

    "📈 <b>Позиция</b>\n"
    f"{position_block}\n"
    "\n"

    "🎯 <b>Сигнал</b>\n"
    f"{signal_block}\n"
    "\n"

    "📐 <b>SAR</b>\n"
    f"{sar_block}\n"
    "\n"

    "➡️ <b>Действие</b>\n"
    f"<b>{action}</b>\n"
    "\n"

    f"⏱ {updated_msk.strftime('%H:%M:%S')} MSK"
)


# ============================================================
# SEND
# ============================================================

url = (
    f"https://api.telegram.org/bot"
    f"{BOT_TOKEN}/sendMessage"
)

payload = {
    "chat_id": TELEGRAM_CHAT_ID,
    "text": telegram_message,
    "parse_mode": "HTML",
}

response = requests.post(
    url,
    json=payload,
    timeout=10
)

response.raise_for_status()

result = response.json()

if not result.get("ok"):

    raise RuntimeError(
        f"Telegram API error: {result}"
    )

print("✅ FINAL STATUS отправлен в Telegram")

✅ FINAL STATUS отправлен в Telegram
